# Central Tendency
**Topic:** Descriptive Statistics

In [1]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go
import plotly.express as px

import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual
from ipywidgets import IntSlider, FloatSlider, Dropdown, Button, Output, HBox, VBox, Label

from IPython.display import display, HTML, clear_output
from scipy import stats
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Calculate** the mean, median, and mode of a dataset
- **Explain** why the mean and median diverge when extreme values are present
- **Interpret** which measure of center is most appropriate for a given distribution shape

> **Tip:** Start by moving the **outlier value slider** to a very high number, watch the mean chase the outlier while the median barely moves.

---
## How we got here

In *01–03* we learned probability, how to measure the chance that something happens. Now we shift to **descriptive statistics**: summarizing data we have already collected. The first question about any dataset is always "where is the center?", and this notebook shows why that question has more than one answer.

---
## Why this matters for data science

Choosing the right measure of center matters enormously when building features, evaluating models, or reporting results. A model trained to minimize mean squared error is implicitly optimizing for the mean. Using mean income to describe a population gives a misleading picture when wealth is concentrated, the median tells a truer story. Knowing when each measure is appropriate is a foundational data science skill.

---
## Try it yourself

In [2]:
from tkh_utils import PALETTE, FONT, base_layout
from tkh_utils import make_output, make_int_slider, make_dropdown, display_widget, register_observer
from IPython.display import display, clear_output
import numpy as np
import plotly.graph_objects as go
from ipywidgets import VBox

# ── Fixed base datasets (seeded for reproducibility) ──────────────────────────
np.random.seed(42)
_BASE = {
    "symmetric":    np.round(np.random.normal(50, 8, 80)).astype(float),
    "right_skewed": np.round(10 + np.random.exponential(scale=15, size=80)).astype(float),
    "bimodal":      np.round(np.concatenate([
                        np.random.normal(30, 5, 40),
                        np.random.normal(70, 5, 40),
                    ])).astype(float),
}
_LABELS = {"symmetric": "Symmetric", "right_skewed": "Right-skewed", "bimodal": "Bimodal"}

out = make_output()

preset_dd = make_dropdown(
    options=[("Symmetric", "symmetric"), ("Right-skewed", "right_skewed"), ("Bimodal", "bimodal")],
    value="symmetric",
    description="Distribution:",
)
outlier_slider = make_int_slider(
    min_val=55, max_val=300, step=5, value=55,
    description="Outlier value:",
)

def _mode_approx(data):
    counts, edges = np.histogram(data, bins=15)
    idx = np.argmax(counts)
    return (edges[idx] + edges[idx + 1]) / 2

def render(preset, outlier_val):
    base = _BASE[preset].copy()
    data = np.append(base, float(outlier_val))

    mean_val   = np.mean(data)
    median_val = np.median(data)
    mode_val   = _mode_approx(base)

    x_max    = int(outlier_val) + 25
    bin_size = max(1, round(x_max / 28))

    counts_arr, _ = np.histogram(data, bins=np.arange(0, x_max + bin_size, bin_size))
    y_top = max(counts_arr) * 1.3 if len(counts_arr) > 0 else 20

    traces = [
        go.Histogram(
            x=data,
            xbins=dict(start=0, end=x_max, size=bin_size),
            marker_color=PALETTE["primary"],
            opacity=0.65,
            name="Data",
        ),
        go.Scatter(
            x=[mean_val, mean_val], y=[0, y_top],
            mode="lines",
            line=dict(color=PALETTE["primary"], width=2.5, dash="dash"),
            name=f"Mean: {mean_val:.1f}",
        ),
        go.Scatter(
            x=[median_val, median_val], y=[0, y_top],
            mode="lines",
            line=dict(color=PALETTE["secondary"], width=2.5, dash="dash"),
            name=f"Median: {median_val:.1f}",
        ),
        go.Scatter(
            x=[mode_val, mode_val], y=[0, y_top],
            mode="lines",
            line=dict(color=PALETTE["accent"], width=2.5, dash="dash"),
            name=f"Mode ≈ {mode_val:.1f}",
        ),
    ]

    layout = base_layout(
        title=f"{_LABELS[preset]} Distribution — Outlier at {outlier_val}",
        xaxis_title="Value",
        yaxis_title="Count",
    )
    layout.update(
        xaxis=dict(range=[0, x_max]),
        yaxis=dict(range=[0, y_top]),
        showlegend=True,
    )

    fig = go.Figure(data=traces, layout=layout)
    with out:
        clear_output(wait=True)
        display(go.FigureWidget(fig))

def on_change(change):
    render(preset_dd.value, outlier_slider.value)

register_observer([preset_dd, outlier_slider], on_change)

display_widget(VBox([preset_dd, outlier_slider]), out)
render(preset_dd.value, outlier_slider.value)


---
## What's happening?

Central tendency describes where the "middle" of a dataset is, but there are three different definitions of middle, and they don't always agree.

| Measure | Symbol | What it controls |
|---------|--------|-----------------|
| Mean | x̄ or μ | Arithmetic average: sum divided by count |
| Median | M | Middle value when data is sorted |
| Mode | — | Most frequently occurring value |

$$\bar{x} = \frac{1}{n} \sum_{i=1}^{n} x_i$$

### The outlier sensitivity rule

The mean uses every value in its calculation, so a single extreme value pulls it in that direction. The median only cares about rank order, so it's resistant to outliers. This is why economists report **median household income** rather than mean: a handful of billionaires would drag the mean far above what a typical household earns.

Look back at the widget with the outlier slider at its maximum, the mean drifts far right while the median holds its ground.

---
## Real-world example: Household income in the United States

Income data is the canonical example of why mean and median diverge. The US income distribution has a long right tail, most households earn moderate amounts, but a small number earn enormous sums that pull the mean upward.

The chart below simulates a realistic income distribution and marks both the mean and median. Notice:

- **Notice:** The peak of the distribution (the mode) is well below both the mean and the median, most households earn less than the average
- **Notice:** The mean is pulled to the right of the median by the long tail of high earners
- **Notice:** The gap between mean and median is itself a measure of inequality, a wider gap signals more skew

> **Discussion question:** If you were advising a city on whether its residents could afford a new housing development priced at the "average" rent, would you use the mean or median income? What could go wrong with the other choice?

In [3]:
np.random.seed(21)

# ── Realistic US household income simulation (2023 approximation, in thousands USD) ──
# Mixture of lognormal components to reproduce the right-skewed shape
n       = 2000
incomes = np.concatenate([
    np.random.lognormal(mean=3.8, sigma=0.6, size=1600),   # middle class cluster
    np.random.lognormal(mean=5.2, sigma=0.8, size=350),    # upper middle
    np.random.lognormal(mean=7.0, sigma=0.5, size=50),     # high earners
])
incomes = np.clip(incomes, 10, 1500)

mean_inc   = np.mean(incomes)
median_inc = np.median(incomes)

fig = go.Figure()
fig.add_trace(go.Histogram(
    x=incomes, nbinsx=60,
    marker_color=PALETTE["primary"], opacity=0.7,
    name="Household income",
))
fig.add_vline(x=mean_inc, line_color=PALETTE["secondary"], line_width=2.5,
              annotation_text=f"Mean: ${mean_inc:.0f}k", annotation_position="top right")
fig.add_vline(x=median_inc, line_color=PALETTE["accent"], line_width=2.5,
              annotation_text=f"Median: ${median_inc:.0f}k", annotation_position="top left")
layout = base_layout(
    title="US Household Income Distribution (simulated, 2023 approximation)",
    xaxis_title="Annual Household Income ($000s)",
    yaxis_title="Number of Households",
)
layout.update(xaxis=dict(range=[0, 600]))
fig.update_layout(layout)
fig.show()

### When to use each measure of center

| Situation | Best measure | Reason |
|-----------|-------------|--------|
| Symmetric, no outliers (e.g., heights) | Mean | Uses all information efficiently |
| Skewed or has outliers (e.g., incomes) | Median | Resistant to extreme values |
| Categorical data (e.g., most common product) | Mode | Mean/median undefined for categories |
| Bimodal data (e.g., two customer segments) | Report both peaks | Single summary hides the structure |
| Setting a loss function in ML | Mean | MSE optimizes the mean; MAE optimizes the median |

---
## Key takeaway

> **The mean uses every value and is pulled by outliers; the median uses only rank and is resistant to them — which one you report is a choice about what story you want the data to tell.**

---
*Next up: Dispersion — once we know where the center is, how spread out are the values around it?*